# German Podcast to English Podcast (Runpod, Hugging Face only)

This notebook runs end-to-end on a Runpod instance with 1 RTX 5090 (32GB VRAM), 31GB RAM, and 12 CPUs:
1. Read a German MP3 podcast
2. Transcribe with diarisation
3. Translate to English
4. Generate a multi-speaker English podcast
5. Save the translated MP3


## Model selection (for RTX 5090 and <60 minutes target)

- **ASR**: `primeline/whisper-large-v3-german` (German-finetuned Whisper with lowest WER — 2.64% avg on CV19/MLS/Tuda-De vs 3.37% for base whisper-large-v3; drop-in replacement via WhisperX)
- **Diarisation**: `pyannote/speaker-diarization-community-1` (pyannote.audio 4.0 successor to 3.1 — 1–4 pp DER improvement across all benchmarks; CC-BY-4.0 license)
- **Translation**: `google/translategemma-12b-it` (TranslateGemma — purpose-built translation model released Jan 2026; outperforms NLLB-200 and even Gemma 3 27B on COMET/MetricX; handles conversational context; fits RTX 5090 at FP16 ~22 GB)
- **TTS**: `SWivid/F5-TTS` (flow-matching TTS with zero-shot voice cloning; Apache 2.0; ~6 GB VRAM; RTF 0.15; replaces XTTS-v2 whose parent company Coqui AI shut down Jan 2024)

### Alternatives considered

| Stage | Runner-up | Notes |
|-------|-----------|-------|
| ASR | `Qwen/Qwen3-ASR-1.7B` | SOTA accuracy (Jan 2026), beats whisper-large-v3 across benchmarks, built-in forced aligner, only ~5 GB VRAM. Requires new pipeline code (not WhisperX-compatible). |
| ASR | `nvidia/canary-1b-v2` | 25 European languages, 749 RTFx, CC-BY-4.0. Requires NeMo framework. |
| Diarisation | `BUT-FIT/diarizen-wavlm-large-s80-md-v2` | Lower DER than pyannote on several benchmarks (e.g. AMI-SDM 13.9% vs 19.9%), but CC-BY-NC license and custom pipeline. |
| Translation | `ByteDance-Seed/Seed-X-PPO-7B` | Competes with GPT-4o quality at 7B; OpenMDW (MIT-like) license; only ~14 GB VRAM. |
| Translation | `Unbabel/Tower-Plus-9B` | Best document-level translation (good for podcasts); CC-BY-NC-SA license (non-commercial). |
| TTS | `ResembleAI/chatterbox` | MIT license, beats ElevenLabs in blind tests, emotion control. |
| TTS | `nari-labs/Dia-1.6B` | Apache 2.0, purpose-built for multi-speaker dialogue with `[S1]`/`[S2]` tags. English only. |
| TTS | `FunAudioLLM/CosyVoice2-0.5B` | Apache 2.0, SOTA speaker similarity (78%), 150ms streaming latency. |

Before running: accept model terms on Hugging Face for `pyannote/speaker-diarization-community-1` and set your read token in `HF_TOKEN`. Place your German MP3 at `INPUT_AUDIO_PATH`.

In [ ]:
%pip -q install whisperx==3.8.1 pyannote.audio==4.0.4 transformers==5.2.0 accelerate sentencepiece f5-tts==1.1.16 pydub==0.25.1

In [ ]:
import os
import torch
import whisperx
import pandas as pd
from pathlib import Path
from pydub import AudioSegment
from transformers import AutoTokenizer, AutoModelForCausalLM
from f5_tts.api import F5TTS

HF_TOKEN = os.environ["HF_TOKEN"]
INPUT_AUDIO_PATH = "german_podcast.mp3"
OUTPUT_AUDIO_PATH = "translated_podcast_en.mp3"
audio_path = INPUT_AUDIO_PATH
device = "cuda"

In [ ]:
audio = whisperx.load_audio(audio_path)
asr_model = whisperx.load_model(
    "primeline/whisper-large-v3-german",
    device,
    compute_type="float16",
    language="de"
)
asr_result = asr_model.transcribe(audio, batch_size=32)
align_model, align_metadata = whisperx.load_align_model(language_code="de", device=device)
aligned_result = whisperx.align(asr_result["segments"], align_model, align_metadata, audio, device)
diarize_model = whisperx.DiarizationPipeline(
    model_name="pyannote/speaker-diarization-community-1",
    use_auth_token=HF_TOKEN,
    device=device
)
diarize_segments = diarize_model(audio)
diarized_result = whisperx.assign_word_speakers(diarize_segments, aligned_result)
segments_df = pd.DataFrame(diarized_result["segments"])[["start", "end", "speaker", "text"]]
segments_df["speaker"] = segments_df["speaker"].fillna("SPEAKER_00")

In [ ]:
model_id = "google/translategemma-12b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
translate_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN
)

translated_texts = []
for text in segments_df["text"].tolist():
    prompt = f"Translate the following German text to English.\nGerman: {text}\nEnglish:"
    inputs = tokenizer(prompt, return_tensors="pt").to(translate_model.device)
    with torch.no_grad():
        outputs = translate_model.generate(**inputs, max_new_tokens=512)
    decoded = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    translated_texts.append(decoded.strip())

segments_df["text_en"] = translated_texts
segments_df.to_csv("podcast_transcript_en.csv", index=False)

del translate_model, tokenizer
torch.cuda.empty_cache()

In [ ]:
work_dir = Path("translated_segments")
work_dir.mkdir(exist_ok=True)
source_audio = AudioSegment.from_file(audio_path)
speaker_refs = segments_df.sort_values("start").drop_duplicates("speaker")[["speaker", "start", "end"]]
speaker_refs["ref_path"] = [work_dir / f"{speaker}_ref.wav" for speaker in speaker_refs["speaker"]]
for row in speaker_refs.itertuples(index=False):
    source_audio[int(row.start * 1000):int(row.end * 1000)].export(row.ref_path, format="wav")
speaker_ref_map = dict(zip(speaker_refs["speaker"], speaker_refs["ref_path"]))

f5tts = F5TTS(model="F5TTS_v1_Base")

segment_paths = []
for row in segments_df.itertuples(index=False):
    segment_path = work_dir / f"seg_{int(row.start * 1000):09d}.wav"
    ref_path = str(speaker_ref_map[row.speaker])
    ref_text = f5tts.transcribe(ref_audio=ref_path)
    f5tts.infer(
        ref_file=ref_path,
        ref_text=ref_text,
        gen_text=row.text_en,
        file_wave=str(segment_path),
    )
    segment_paths.append(segment_path)

In [ ]:
translated_audio = AudioSegment.silent(duration=0)
cursor_ms = 0
for row, segment_path in zip(segments_df.itertuples(index=False), segment_paths):
    start_ms = int(row.start * 1000)
    translated_audio = translated_audio + AudioSegment.silent(duration=max(start_ms - cursor_ms, 0))
    segment_audio = AudioSegment.from_wav(segment_path)
    translated_audio = translated_audio + segment_audio
    cursor_ms = len(translated_audio)
translated_audio.export(OUTPUT_AUDIO_PATH, format="mp3")
